# WP2 T2.4 DBRepo View Creation

This notebook creates/registers the T2.4 DBRepo views used by the T2.5 verification notebook. It reads the source definitions from `src/sql/04_views.sql`, reads DBRepo credentials from `.env`, checks the existing DBRepo views, then submits the four expected views to the DBRepo REST API.

DBRepo's documented REST endpoint is `POST /api/v1/database/{databaseId}/view`. The documentation describes a structured query payload based on column IDs, joins, filters, and orders. This notebook uses that documented shape for the join-only ML feature view and also tries raw-SQL payload variants for the aggregate views, because the public structured payload does not expose aggregate/group-by/union fields.

In [19]:
from pathlib import Path
import os
import re
import requests
import pandas as pd
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import (
    QueryDefinition,
    JoinDefinition,
    JoinType,
    ConditionalDefinition,
    CreateView,
    Subset,
    SubsetColumn,
    Join,
    Conditional,
)

try:
    from dotenv import load_dotenv
    load_dotenv(override=True)
except ImportError as exc:
    raise ImportError("Install python-dotenv in this notebook kernel: %pip install python-dotenv") from exc

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
SOURCE_SQL_PATH = PROJECT_ROOT / "src" / "sql" / "04_views.sql"
SOURCE_SQL = SOURCE_SQL_PATH.read_text(encoding="utf-8")

DBREPO_BASE_URL = os.getenv("DBREPO_BASE_URL", "https://test.dbrepo.tuwien.ac.at").rstrip("/")
DBREPO_USER = os.getenv("DBREPO_USER")
DBREPO_PASSWORD = os.getenv("DBREPO_PASSWORD")
DBREPO_DB_ID = os.getenv("DBREPO_DB_ID", "3d81c073-e5fd-49b9-9536-b75ed490ca3e")
API_BASE = f"{DBREPO_BASE_URL}/api/v1"
AUTH = (DBREPO_USER, DBREPO_PASSWORD)
CLIENT = RestClient(DBREPO_BASE_URL, username=DBREPO_USER, password=DBREPO_PASSWORD)

CREATE_DBREPO_VIEWS = os.getenv("CREATE_DBREPO_VIEWS", "true").lower() == "true"
REPLACE_EXISTING_VIEWS = os.getenv("REPLACE_EXISTING_VIEWS", "false").lower() == "true"

if not DBREPO_USER or not DBREPO_PASSWORD:
    raise RuntimeError("Set DBREPO_USER and DBREPO_PASSWORD in .env before running this notebook.")

print(f"DBRepo API: {API_BASE}")
print(f"Database: {DBREPO_DB_ID}")
print(f"CREATE_DBREPO_VIEWS={CREATE_DBREPO_VIEWS}")
print(f"REPLACE_EXISTING_VIEWS={REPLACE_EXISTING_VIEWS}")
print(f"Source SQL: {SOURCE_SQL_PATH}")

DBRepo API: https://test.dbrepo.tuwien.ac.at/api/v1
Database: 3d81c073-e5fd-49b9-9536-b75ed490ca3e
CREATE_DBREPO_VIEWS=True
REPLACE_EXISTING_VIEWS=False
Source SQL: C:\Users\maxis\iCloudDrive\Master\Data Stewardship\Übung\part 3\dast-2026-crash-severity-prediction\src\sql\04_views.sql


## Helpers and Metadata Lookup

In [20]:
def request_json(method, path, expected=(200,), **kwargs):
    response = requests.request(
        method,
        f"{API_BASE}{path}",
        auth=AUTH,
        headers={"Accept": "application/json", **kwargs.pop("headers", {})},
        timeout=120,
        **kwargs,
    )
    if response.status_code not in expected:
        raise RuntimeError(f"{method} {path} failed: {response.status_code} {response.text}")
    if not response.text:
        return None
    return response.json()

def list_tables():
    tables = request_json("GET", f"/database/{DBREPO_DB_ID}/table")
    return {table["name"]: table for table in tables}

def get_table_details(table_lookup):
    details = {}
    for table_name, table in table_lookup.items():
        details[table_name] = request_json("GET", f"/database/{DBREPO_DB_ID}/table/{table['id']}")
    return details

def list_views():
    views = request_json("GET", f"/database/{DBREPO_DB_ID}/view")
    return {view["name"]: view for view in views}

def column_id(table_details, table_name, column_name):
    for column in table_details[table_name].get("columns", []):
        if column.get("name") == column_name:
            return column["id"]
    raise KeyError(f"Column not found: {table_name}.{column_name}")

table_lookup = list_tables()
required_tables = {"collision", "vehicle", "casualty"}
missing_tables = required_tables - set(table_lookup)
if missing_tables:
    raise RuntimeError(f"Missing required DBRepo tables: {sorted(missing_tables)}")

table_details = get_table_details({name: table_lookup[name] for name in required_tables})
existing_views = list_views()

print("Tables:", sorted(required_tables))
print("Existing views:", sorted(existing_views))

Tables: ['casualty', 'collision', 'vehicle']
Existing views: []


## View Definitions

The SQL statements below are equivalent to `src/sql/04_views.sql`, with PostgreSQL-only syntax rewritten to MariaDB-compatible expressions where needed.

In [21]:
VIEW_SQL = {
    "v_ml_features": """
SELECT
    cas.casualty_id,
    c.collision_index,
    cas.casualty_severity,
    c.road_type,
    c.speed_limit,
    c.weather_conditions,
    c.light_conditions,
    c.road_surface_conditions,
    c.time,
    c.day_of_week,
    c.number_of_vehicles,
    v.vehicle_type
FROM casualty cas
JOIN collision c ON cas.collision_index = c.collision_index
JOIN vehicle v ON cas.collision_index = v.collision_index
    AND cas.vehicle_reference = v.vehicle_reference
""".strip(),
    "v_severity_distribution": """
SELECT
    casualty_severity,
    COUNT(*) AS total_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM casualty
GROUP BY casualty_severity
ORDER BY total_count DESC
""".strip(),
    "v_collision_summary": """
SELECT
    c.collision_index,
    c.date,
    c.day_of_week,
    c.time,
    c.road_type,
    c.speed_limit,
    c.weather_conditions,
    c.light_conditions,
    c.road_surface_conditions,
    c.number_of_vehicles,
    c.number_of_casualties,
    CASE
        WHEN MAX(cas.casualty_severity = 'Fatal') = 1 THEN 'Fatal'
        WHEN MAX(cas.casualty_severity = 'Serious') = 1 THEN 'Serious'
        ELSE 'Slight'
    END AS worst_severity
FROM collision c
JOIN casualty cas ON c.collision_index = cas.collision_index
GROUP BY
    c.collision_index, c.date, c.day_of_week, c.time,
    c.road_type, c.speed_limit, c.weather_conditions,
    c.light_conditions, c.road_surface_conditions,
    c.number_of_vehicles, c.number_of_casualties
""".strip(),
    "v_feature_null_check": """
SELECT 'road_type' AS feature, SUM(road_type IS NULL) AS null_count, COUNT(*) AS total FROM collision
UNION ALL SELECT 'speed_limit', SUM(speed_limit IS NULL), COUNT(*) FROM collision
UNION ALL SELECT 'weather_conditions', SUM(weather_conditions IS NULL), COUNT(*) FROM collision
UNION ALL SELECT 'light_conditions', SUM(light_conditions IS NULL), COUNT(*) FROM collision
UNION ALL SELECT 'road_surface_conditions', SUM(road_surface_conditions IS NULL), COUNT(*) FROM collision
UNION ALL SELECT 'time', SUM(time IS NULL), COUNT(*) FROM collision
UNION ALL SELECT 'day_of_week', SUM(day_of_week IS NULL), COUNT(*) FROM collision
UNION ALL SELECT 'number_of_vehicles', SUM(number_of_vehicles IS NULL), COUNT(*) FROM collision
UNION ALL SELECT 'vehicle_type', SUM(vehicle_type IS NULL), COUNT(*) FROM vehicle
UNION ALL SELECT 'casualty_severity', SUM(casualty_severity IS NULL), COUNT(*) FROM casualty
""".strip(),
}

display(pd.DataFrame([{"view": name, "sql": sql} for name, sql in VIEW_SQL.items()]))

,view,sql
0,v_ml_features,"SELECT\n cas.casualty_id,\n c.collision_index,\n cas.casualty_severity,\n c.road_type,\n c.speed_limit,\n c.weather_conditions,\n c.light_conditions,\n c.road_surface_conditions,\n c.time,\n c.day_of_week,\n c.number_of_vehicles,\n v.vehicle_type\nFROM casualty cas\nJOIN collision c ON cas.collision_index = c.collision_index\nJOIN vehicle v ON cas.collision_index = v.collision_index\n AND cas.vehicle_reference = v.vehicle_reference"
1,v_severity_distribution,"SELECT\n casualty_severity,\n COUNT(*) AS total_count,\n ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage\nFROM casualty\nGROUP BY casualty_severity\nORDER BY total_count DESC"
2,v_collision_summary,"SELECT\n c.collision_index,\n c.date,\n c.day_of_week,\n c.time,\n c.road_type,\n c.speed_limit,\n c.weather_conditions,\n c.light_conditions,\n c.road_surface_conditions,\n c.number_of_vehicles,\n c.number_of_casualties,\n CASE\n WHEN MAX(cas.casualty_severity = 'Fatal') = 1 THEN 'Fatal'\n WHEN MAX(cas.casualty_severity = 'Serious') = 1 THEN 'Serious'\n ELSE 'Slight'\n END AS worst_severity\nFROM collision c\nJOIN casualty cas ON c.collision_index = cas.collision_index\nGROUP BY\n c.collision_index, c.date, c.day_of_week, c.time,\n c.road_type, c.speed_limit, c.weather_conditions,\n c.light_conditions, c.road_surface_conditions,\n c.number_of_vehicles, c.number_of_casualties"
3,v_feature_null_check,"SELECT 'road_type' AS feature, SUM(road_type IS NULL) AS null_count, COUNT(*) AS total FROM collision\nUNION ALL SELECT 'speed_limit', SUM(speed_limit IS NULL), COUNT(*) FROM collision\nUNION ALL SELECT 'weather_conditions', SUM(weather_conditions IS NULL), COUNT(*) FROM collision\nUNION ALL SELECT 'light_conditions', SUM(light_conditions IS NULL), COUNT(*) FROM collision\nUNION ALL SELECT 'road_surface_conditions', SUM(road_surface_conditions IS NULL), COUNT(*) FROM collision\nUNION ALL SELECT 'time', SUM(time IS NULL), COUNT(*) FROM collision\nUNION ALL SELECT 'day_of_week', SUM(day_of_week IS NULL), COUNT(*) FROM collision\nUNION ALL SELECT 'number_of_vehicles', SUM(number_of_vehicles IS NULL), COUNT(*) FROM collision\nUNION ALL SELECT 'vehicle_type', SUM(vehicle_type IS NULL), COUNT(*) FROM vehicle\nUNION ALL SELECT 'casualty_severity', SUM(casualty_severity IS NULL), COUNT(*) FROM casualty"


## Official Python Client Query for `v_ml_features`

DBRepo's Python client maps `QueryDefinition` objects to the documented REST `CreateViewDto`. The installed client supports selected columns, datasources, joins, filters, and orders. It does not expose SQL aggregation, `GROUP BY`, calculated columns, or `UNION`.

In [22]:
def qcol(table_name, column_name):
    return f"{table_name}.{column_name}"

def subset_col(table_name, column_name):
    return SubsetColumn(id=column_id(table_details, table_name, column_name))

def subset_condition(left_table, left_column, right_table, right_column):
    return Conditional(
        column_id=column_id(table_details, left_table, left_column),
        foreign_column_id=column_id(table_details, right_table, right_column),
    )

LOW_LEVEL_CREATE_VIEW_PAYLOADS = {
    "v_ml_features": CreateView(
        name="v_ml_features",
        query=Subset(
            datasource_ids=[table_lookup["casualty"]["id"]],
            columns=[
                subset_col("casualty", "casualty_id"),
                subset_col("collision", "collision_index"),
                subset_col("casualty", "casualty_severity"),
                subset_col("collision", "road_type"),
                subset_col("collision", "speed_limit"),
                subset_col("collision", "weather_conditions"),
                subset_col("collision", "light_conditions"),
                subset_col("collision", "road_surface_conditions"),
                subset_col("collision", "time"),
                subset_col("collision", "day_of_week"),
                subset_col("collision", "number_of_vehicles"),
                subset_col("vehicle", "vehicle_type"),
            ],
            joins=[
                Join(
                    type=JoinType.INNER,
                    datasource_id=table_lookup["collision"]["id"],
                    conditionals=[subset_condition("casualty", "collision_index", "collision", "collision_index")],
                ),
                Join(
                    type=JoinType.INNER,
                    datasource_id=table_lookup["vehicle"]["id"],
                    conditionals=[
                        subset_condition("casualty", "collision_index", "vehicle", "collision_index"),
                        subset_condition("casualty", "vehicle_reference", "vehicle", "vehicle_reference"),
                    ],
                ),
            ],
            filters=[],
            orders=[],
        ),
        is_public=True,
        is_schema_public=True,
    )
}

PYTHON_CLIENT_QUERIES = {
    "v_ml_features": QueryDefinition(
        datasources=["casualty"],
        columns=[
            qcol("casualty", "casualty_id"),
            qcol("collision", "collision_index"),
            qcol("casualty", "casualty_severity"),
            qcol("collision", "road_type"),
            qcol("collision", "speed_limit"),
            qcol("collision", "weather_conditions"),
            qcol("collision", "light_conditions"),
            qcol("collision", "road_surface_conditions"),
            qcol("collision", "time"),
            qcol("collision", "day_of_week"),
            qcol("collision", "number_of_vehicles"),
            qcol("vehicle", "vehicle_type"),
        ],
        joins=[
            JoinDefinition(
                type=JoinType.INNER,
                datasource="collision",
                conditionals=[
                    ConditionalDefinition(
                        column=qcol("casualty", "collision_index"),
                        foreign_column=qcol("collision", "collision_index"),
                    )
                ],
            ),
            JoinDefinition(
                type=JoinType.INNER,
                datasource="vehicle",
                conditionals=[
                    ConditionalDefinition(
                        column=qcol("casualty", "collision_index"),
                        foreign_column=qcol("vehicle", "collision_index"),
                    ),
                    ConditionalDefinition(
                        column=qcol("casualty", "vehicle_reference"),
                        foreign_column=qcol("vehicle", "vehicle_reference"),
                    ),
                ],
            ),
        ],
    )
}

LOW_LEVEL_CREATE_VIEW_PAYLOADS["v_ml_features"]

CreateView(name='v_ml_features', query=Subset(columns=[SubsetColumn(id='00c78c05-af20-4967-a66e-c0428fdb95a7', alias=None), SubsetColumn(id='e00c6291-b0bd-4f0a-a1bc-16dada492e87', alias=None), SubsetColumn(id='36488f1e-2af9-4411-ae2d-897ad478700e', alias=None), SubsetColumn(id='22237778-0372-4f08-9878-86988182e598', alias=None), SubsetColumn(id='ad009692-3f6b-45ec-adaa-4b43f8e6e662', alias=None), SubsetColumn(id='94605670-95d9-48f6-9f6a-8dcc88bdb3e1', alias=None), SubsetColumn(id='7d042d9c-bf92-4efe-a16d-1702fa20d09d', alias=None), SubsetColumn(id='2c6b9358-a614-4aa0-86ff-c5eee9ee4a56', alias=None), SubsetColumn(id='f838d7e1-204e-46d6-8ebd-7f17a7b30181', alias=None), SubsetColumn(id='12fc8eff-0a7a-40e1-8dc1-b297f77b8151', alias=None), SubsetColumn(id='619b5355-1391-45ec-b9da-0d4706e83108', alias=None), SubsetColumn(id='355bd008-013d-4d5b-852e-ed567e8d66e7', alias=None)], datasource_ids=['9f0bc3f2-2589-4631-84cb-9220ba3e22bf'], joins=[Join(type=<JoinType.INNER: 'inner'>, datasource_id='

## Create Views

For `v_ml_features`, the notebook uses the official DBRepo Python client and `QueryDefinition`. For the aggregate views, it tries raw SQL payload variants as a best-effort fallback because DBRepo's public `CreateViewDto`/`QueryDefinition` model does not expose aggregate/group-by/union fields. If those raw SQL variants are rejected by your DBRepo instance, those SQL views cannot be created through the documented DBRepo view API.

In [23]:
pd.set_option("display.max_colwidth", None)

def delete_view(view):
    CLIENT.delete_view(database_id=DBREPO_DB_ID, view_id=view["id"])

def create_view_with_client(view_name, query):
    return CLIENT.create_view(
        database_id=DBREPO_DB_ID,
        name=view_name,
        query=query,
        is_public=True,
        is_schema_public=True,
    )

def create_view_payload(payload):
    if hasattr(payload, "model_dump"):
        payload = payload.model_dump(mode="json")
    return request_json(
        "POST",
        f"/database/{DBREPO_DB_ID}/view",
        expected=(201,),
        json=payload,
        headers={"Content-Type": "application/json"},
    )

def raw_sql_payload_variants(view_name, sql):
    return [
        {"name": view_name, "query": sql, "is_public": True, "is_schema_public": True},
        {"name": view_name, "query": {"query": sql}, "is_public": True, "is_schema_public": True},
        {"name": view_name, "sql": sql, "is_public": True, "is_schema_public": True},
        {"name": view_name, "statement": sql, "is_public": True, "is_schema_public": True},
    ]

create_rows = []
existing_views = list_views()

for view_name in VIEW_SQL:
    if view_name in existing_views and not REPLACE_EXISTING_VIEWS:
        create_rows.append({"view": view_name, "status": "already_exists", "detail": existing_views[view_name]["id"]})
        continue
    if view_name in existing_views and REPLACE_EXISTING_VIEWS:
        delete_view(existing_views[view_name])

    if not CREATE_DBREPO_VIEWS:
        create_rows.append({"view": view_name, "status": "dry_run", "detail": "CREATE_DBREPO_VIEWS=false"})
        continue

    payloads = []
    low_level_payload = LOW_LEVEL_CREATE_VIEW_PAYLOADS.get(view_name)
    if low_level_payload is not None:
        payloads.append(("low_level_create_view_dto", low_level_payload))
    client_query = PYTHON_CLIENT_QUERIES.get(view_name)
    if client_query is not None:
        try:
            created = create_view_with_client(view_name, client_query)
            create_rows.append({"view": view_name, "status": "created", "detail": f"python_client_query: {created.id}"})
            continue
        except Exception as exc:
            payloads.append(("python_client_query_failed", {"error": repr(exc)}))
    payloads.extend((f"raw_sql_variant_{idx + 1}", payload) for idx, payload in enumerate(raw_sql_payload_variants(view_name, VIEW_SQL[view_name])))

    success = False
    errors = []
    for strategy, payload in payloads:
        try:
            if strategy == "python_client_query_failed":
                raise RuntimeError(payload["error"])
            created = create_view_payload(payload)
            create_rows.append({"view": view_name, "status": "created", "detail": f"{strategy}: {created.get('id')}"})
            success = True
            break
        except Exception as exc:
            errors.append(f"{strategy}: {exc}")
    if not success:
        create_rows.append({"view": view_name, "status": "failed", "detail": " | ".join(errors)})

create_report = pd.DataFrame(create_rows)
display(create_report)

failed = create_report[create_report["status"] == "failed"]
if len(failed):
    print("Some views could not be created through the public DBRepo view API. Full details:")
    for _, row in failed.iterrows():
        print(f"\n{row['view']}:\n{row['detail']}")

,view,status,detail
0,v_ml_features,failed,"low_level_create_view_dto: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 403 {""status"":""FORBIDDEN"",""message"":""Failed to create view: not the database owner"",""code"":""error.request.forbidden""} | python_client_query_failed: ValueError('setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.') | raw_sql_variant_1: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {""type"":""about:blank"",""title"":""Bad Request"",""status"":400,""detail"":""Failed to read request"",""instance"":""/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view"",""properties"":null} | raw_sql_variant_2: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {""type"":""about:blank"",""title"":""Bad Request"",""status"":400,""detail"":""Failed to read request"",""instance"":""/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view"",""properties"":null} | raw_sql_variant_3: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {""type"":""about:blank"",""title"":""Bad Request"",""status"":400,""detail"":""Failed to read request"",""instance"":""/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view"",""properties"":null} | raw_sql_variant_4: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {""type"":""about:blank"",""title"":""Bad Request"",""status"":400,""detail"":""Failed to read request"",""instance"":""/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view"",""properties"":null}"
1,v_severity_distribution,failed,"raw_sql_variant_1: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {""type"":""about:blank"",""title"":""Bad Request"",""status"":400,""detail"":""Failed to read request"",""instance"":""/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view"",""properties"":null} | raw_sql_variant_2: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {""type"":""about:blank"",""title"":""Bad Request"",""status"":400,""detail"":""Failed to read request"",""instance"":""/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view"",""properties"":null} | raw_sql_variant_3: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {""type"":""about:blank"",""title"":""Bad Request"",""status"":400,""detail"":""Failed to read request"",""instance"":""/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view"",""properties"":null} | raw_sql_variant_4: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {""type"":""about:blank"",""title"":""Bad Request"",""status"":400,""detail"":""Failed to read request"",""instance"":""/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view"",""properties"":null}"
2,v_collision_summary,failed,"raw_sql_variant_1: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {""type"":""about:blank"",""title"":""Bad Request"",""status"":400,""detail"":""Failed to read request"",""instance"":""/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view"",""properties"":null} | raw_sql_variant_2: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {""type"":""about:blank"",""title"":""Bad Request"",""status"":400,""detail"":""Failed to read request"",""instance"":""/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view"",""properties"":null} | raw_sql_variant_3: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {""type"":""about:blank"",""title"":""Bad Request"",""status"":400,""detail"":""Failed to read request"",""instance"":""/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view"",""properties"":null} | raw_sql_variant_4: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {""type"":""about:blank"",""title"":""Bad Request"",""status"":400,""detail"":""Failed to read request"",""instance"":""/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed4

Some views could not be created through the public DBRepo view API. Full details:

v_ml_features:
low_level_create_view_dto: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 403 {"status":"FORBIDDEN","message":"Failed to create view: not the database owner","code":"error.request.forbidden"} | python_client_query_failed: ValueError('setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.') | raw_sql_variant_1: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {"type":"about:blank","title":"Bad Request","status":400,"detail":"Failed to read request","instance":"/api/v1/database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view","properties":null} | raw_sql_variant_2: POST /database/3d81c073-e5fd-49b9-9536-b75ed490ca3e/view failed: 400 {"type":"about:blank","title":"Bad Request","status":400,"detail":"Failed to read request","instance":"/api/v1/database/3d81c07

## Verify DBRepo View List

In [24]:
views_after = list_views()
expected_view_names = set(VIEW_SQL)
verification = pd.DataFrame([
    {
        "view": view_name,
        "present_in_dbrepo": view_name in views_after,
        "view_id": views_after.get(view_name, {}).get("id"),
        "query": views_after.get(view_name, {}).get("query"),
    }
    for view_name in sorted(expected_view_names)
])
display(verification)

missing = verification.loc[~verification["present_in_dbrepo"], "view"].tolist()
if missing:
    print("Missing views:", missing)
    print("If these are aggregate views, DBRepo's documented structured API may not support their SQL shape. Create them via DBRepo UI/direct SQL, then rerun this verification cell and dbrepo_load_verify.ipynb.")
else:
    print("All expected T2.4 views are present in DBRepo. You can now rerun dbrepo_load_verify.ipynb with LOAD_TO_DBREPO=false.")

,view,present_in_dbrepo,view_id,query
0,v_collision_summary,False,None,None
1,v_feature_null_check,False,None,None
2,v_ml_features,False,None,None
3,v_severity_distribution,False,None,None


Missing views: ['v_collision_summary', 'v_feature_null_check', 'v_ml_features', 'v_severity_distribution']
If these are aggregate views, DBRepo's documented structured API may not support their SQL shape. Create them via DBRepo UI/direct SQL, then rerun this verification cell and dbrepo_load_verify.ipynb.
